# RFSoC 4x2 – Pulse / Arbitrary Waveform Generator

**What this does:** plays up to **4 waveforms** (2 outputs × 2 slots) from the board's DACs, with
sample-accurate timing (1 sample = 101.7 ps).

**How to use it – 4 steps**
1. **Settings** – choose the length of the waveform window.
2. **Waveforms** – describe your pulses (frequency, power, start time, duration, shape).
3. **Preview** – plot and check levels *before* touching the hardware.
4. **Play** – load the FPGA, upload, start / stop.

Steps 1-3 also run on a normal laptop with `DRY_RUN = True`.

**Hardware limits to remember**

| Item | Value |
|---|---|
| Outputs | `"DAC_A"` and `"DAC_B"` |
| Slots per output | 2 (`"z"` and `"x"`), same length for all |
| Max slot length | 26.67 µs (2¹⁸ samples) |
| Length granularity | 1.6276 ns (16 samples) – rounded automatically |
| Time resolution inside the window | 101.7 ps (1 sample) |
| Max amplitude | ±0.25 V peak ≈ −2 dBm (into 50 Ω); more gets clipped (you get a warning) |

In [ ]:
# Needs rfsoc_awg.py in the same folder as this notebook.
import matplotlib.pyplot as plt
from rfsoc_awg import *      # AWG, Pulse and the unit helpers ns, us, kHz, MHz, GHz

%matplotlib inline

## Step 1 – Settings

In [ ]:
DRY_RUN = False        # True = design/plot only, never touch the board (works on any PC)

# Length of ONE waveform slot. It is rounded to a multiple of 1.6276 ns; the real value is printed.
# (Max 26.67 us.)
awg = AWG(window=6.65 * us,
          bitfile="./awg_clock_syncd.bit",   # the FPGA design
          dry_run=DRY_RUN)

## Step 2 – Describe your waveforms

`awg.set_waveform(output, slot, what_to_play)` where `what_to_play` is a `Pulse`, a list of
`Pulse` objects (they are added together), or your own numpy array in volts.

`Pulse` options (all times in seconds – use `ns`, `us`):

| Argument | Meaning |
|---|---|
| `freq` | carrier frequency, e.g. `70*MHz` |
| `power_dbm` **or** `vpp` | peak amplitude (dBm into 50 Ω, or volts peak-to-peak) |
| `start` | when the pulse begins, counted from the start of the slot (default 0) |
| `duration` | pulse length (default: until the end of the window) |
| `phase_deg` | carrier phase |
| `shape` | `"rect"`, `"gauss"`, `"cos"` (smooth bump) or `"flat"` (flat top with smooth edges, needs `rise=`) |
| `phase_ref` | `"window"` (default; phase tied to the slot start → all pulses coherent) or `"pulse"` (phase restarts at each pulse) |

Below: the same four continuous tones as in the original script (one per output/slot).

In [ ]:
# --- Same signals as the original notebook: full-window sine waves --------------------
awg.set_waveform("DAC_B", "z", Pulse(freq=20 * MHz, power_dbm=-25))
awg.set_waveform("DAC_B", "x", Pulse(freq=40 * MHz, power_dbm=-15))
awg.set_waveform("DAC_A", "z", Pulse(freq=60 * MHz, power_dbm=-20))
awg.set_waveform("DAC_A", "x", Pulse(freq=70 * MHz, power_dbm=-10))

### Pulse examples (copy / adapt – each call replaces what the slot played before)

In [ ]:
# A) A 200 ns Gaussian pulse that starts 1 us after the start of the window
awg.set_waveform("DAC_A", "z",
                 Pulse(freq=70 * MHz, power_dbm=-10, start=1 * us, duration=200 * ns, shape="gauss"))

# B) Two rectangular pulses with different phase (e.g. a pi/2 - wait - pi/2 sequence)
awg.set_waveform("DAC_A", "x", [
    Pulse(freq=70 * MHz, power_dbm=-12, start=0.5 * us, duration=100 * ns),
    Pulse(freq=70 * MHz, power_dbm=-12, start=2.5 * us, duration=100 * ns, phase_deg=90),
])

# C) Flat-top pulse with 20 ns smooth edges (less spectral splatter than a hard rectangle)
awg.set_waveform("DAC_B", "z",
                 Pulse(freq=100 * MHz, vpp=0.1, start=3 * us, duration=1 * us, shape="flat", rise=20 * ns))

# D) Any custom shape: build a numpy array in VOLTS. awg.t is the time axis of the window.
import numpy as np
chirp = 0.1 * np.sin(2 * np.pi * (50 * MHz * awg.t + 0.5 * (20 * MHz / awg.window) * awg.t**2))   # linear sweep 50 -> 70 MHz
awg.set_waveform("DAC_B", "x", chirp)

# E) Empty a slot (outputs zero):  awg.set_waveform("DAC_B", "x", None)

## Step 3 – Preview (no hardware needed)
Always check this before playing. Red dotted lines = DAC full scale.
To inspect individual carrier cycles, zoom: `awg.plot(1*us, 1.05*us)`.

In [ ]:
awg.summary()   # peak level of every slot (+ clipping warning)
awg.plot()
plt.show()

## Step 4 – Play on the board
`connect()` loads the FPGA design and programs the on-board clock chips (≈ seconds, do it once).
`upload()` sends the waveforms to the board (repeat after every change).

In [ ]:
awg.connect()    # load bitstream, program clocks, reset the AWG logic
awg.upload()     # send waveforms + length to the board RAM
awg.start()      # enable the outputs

### Changed a pulse? Re-send and restart
Edit the `set_waveform(...)` calls above, then run this cell (no need to `connect()` again).

In [ ]:
awg.play()       # = stop -> upload -> start

### Stop

In [ ]:
awg.stop()

### Optional – check the board clocks
Expected: CPU ≈ 1200 MHz, FCLK0 ≈ 214 MHz, FCLK1-3 ≈ 100 MHz.

In [ ]:
if not DRY_RUN:
    print_clocks()

## Troubleshooting
* **`pynq`/`xrfclk` import error** – you are not on the board. Use `DRY_RUN = True`.
* **"Pulse ends at … but the window is only …"** – increase `window` in step 1 or shorten the pulse.
* **"Clipping!" warning** – requested level is above the DAC full scale (about −2 dBm). Lower `power_dbm`/`vpp`.
* **Absolute power looks off** – the dBm ↔ code calibration (`VPP_AT_FULL_SCALE` = 0.5 V in `rfsoc_awg.py`) comes from the
  original script; measure with a scope / spectrum analyser and update it if your output chain differs.
* **Nothing at the output** – did you call `awg.start()` after `awg.upload()`? Is the channel you use defined (outputs
  with no waveform are left disabled)?